# Day 5 — Hands-On Lab 1: Transformations + Data Quality + Quarantine

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Source** | `gbmart.bronze.customers` |
| **Target** | `gbmart.silver.customers` (clean) + `gbmart.silver.customers_quarantine` (rejected) |
| **Duration** | 60 minutes |
| **Follows** | ILT 1 (Standardize/Conform/Enrich) + ILT 2 (DQ/Constraints/Quarantine) |

### Learning Objectives
- Run a DQ scan and read its output like a checklist, not just a number
- Apply standardize fixes (email, phone) with re-validation to confirm the fix actually worked
- Make and justify a quarantine decision
- Split clean vs. quarantine, write both tables, and verify
- Add `NOT NULL` and `CHECK` constraints to the Silver table as a permanent backstop behind the DQ scan

---
**Instructions:** Run each cell with **Shift + Enter**. Blanks marked `# YOUR CODE HERE` are for you to complete — the DQ rules and thresholds are given in the comments above each blank.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import DateType
from pyspark.sql.window import Window

# PhoneNumber is BIGINT in Bronze -- disabling ANSI mode stops Spark from
# forcing our formatted string back into a number.
spark.conf.set("spark.sql.ansi.enabled", "false")

CATALOG          = "gbmart"
BRONZE_TABLE     = "gbmart.bronze.customers"
SILVER_TABLE     = "gbmart.silver.customers"
QUARANTINE_TABLE = "gbmart.silver.customers_quarantine"

print(f"Reading from : {BRONZE_TABLE}")
print(f"Writing to   : {SILVER_TABLE}")
print(f"Quarantine   : {QUARANTINE_TABLE}")

## Phase A — First DQ Scan (Observe Only)

Before fixing anything, read the raw data and tag every row with its first failing rule. Don't fix yet — just observe what's actually there.

In [ ]:
bronze_df = spark.table(BRONZE_TABLE)
print(f"Total records in Bronze: {bronze_df.count():,}")
bronze_df.printSchema()

In [ ]:
EMAIL_REGEX = r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'

dq_scan_df = bronze_df \
    .withColumn("_dob_temp",   to_date(col("DateOfBirth"),      "yyyy-MM-dd")) \
    .withColumn("_reg_temp",   to_date(col("RegistrationDate"), "yyyy-MM-dd")) \
    .withColumn("_age_at_reg", floor(datediff(col("_reg_temp"), col("_dob_temp")) / 365.25)) \
    .withColumn("_dq_issue",
        # TASK 1: complete this priority chain.
        # Rules, in priority order (first match wins):
        #   1. CustomerID is null                       -> "NULL_CUSTOMER_ID"
        #   2. FirstName is null or blank after trim     -> "NULL_FIRST_NAME"
        #   3. Email is null                             -> "NULL_EMAIL"
        #   4. Email does not match EMAIL_REGEX          -> "INVALID_EMAIL_FORMAT"
        #   5. age at registration < 18                  -> "REGISTERED_UNDER_18"
        #   6. otherwise                                 -> None (clean row)
        when(col("CustomerID").isNull(),                                lit("NULL_CUSTOMER_ID"))
        .when(col("FirstName").isNull() | (trim(col("FirstName")) == ""), lit("NULL_FIRST_NAME"))
        .when(col("Email").isNull(),                                    lit("NULL_EMAIL"))
        # YOUR CODE HERE: add the INVALID_EMAIL_FORMAT rule (rule 4 above)
        .when(col("_age_at_reg") < 18,                                  lit("REGISTERED_UNDER_18"))
        .otherwise(lit(None))
    )

print("=== DQ Issues Found ===")
dq_scan_df.groupBy("_dq_issue").count().orderBy("count", ascending=False).show()

**Q: How many distinct `_dq_issue` categories showed up, and which one has the most rows?**

*Your answer:* _______________

## Phase B — Investigate Before Fixing

Look at the actual invalid emails before writing a fix. Guessing the fix without looking at real examples is how you end up "fixing" data that wasn't broken.

In [ ]:
dq_scan_df.filter(col("_dq_issue") == "INVALID_EMAIL_FORMAT") \
    .select("CustomerID", "Email") \
    .show(20, truncate=False)

You should see two patterns: a stray space inserted mid-email (mobile autocorrect), and a `d'` prefix from surnames like D'Souza/D'Silva that most email providers strip but this form accepted.

**Business decision: fix, don't quarantine.** Both are recoverable formatting errors — quarantining them would mean losing real customers over a keyboard quirk.

In [ ]:
# TASK 2: apply both email fixes, then re-validate.
# Fix 1: remove any stray spaces
# Fix 2: remove the pattern  d  +  any apostrophe variant ('\u2019\u2018)
remediated_df = bronze_df.withColumn(
    "Email",
    regexp_replace(
        regexp_replace(
            trim(lower(col("Email"))),
            " ", ""                       # remove spaces
        ),
        # YOUR CODE HERE: remove d + apostrophe pattern -- regex is  d['\u2019\u2018]
        "", ""
    )
)

# Re-validate -- this MUST show 0 for the fix to be considered correct
still_invalid = remediated_df.filter(~col("Email").rlike(EMAIL_REGEX) & col("Email").isNotNull()).count()
print(f"Remaining invalid emails after fix: {still_invalid}  (expected: 0)")

## Phase C — Standardize: Phone Number Format

Check the length distribution of `PhoneNumber` before assuming anything about its format.

In [ ]:
remediated_df \
    .withColumn("_phone_len", length(col("PhoneNumber").cast("string"))) \
    .groupBy("_phone_len").count() \
    .orderBy("_phone_len") \
    .show()

**Q: What length is every phone number, and what does that tell you about the source form's behavior?**

*Your answer:* _______________

In [ ]:
# TASK 3: reformat 12-digit numbers starting with '91' into '+91-XXXXXXXXXX'.
# Only touch numbers that actually match this exact shape -- never blindly
# slice a string without confirming its length and prefix first.
remediated_df = remediated_df \
    .withColumn("PhoneNumber", col("PhoneNumber").cast("string")) \
    .withColumn(
        "PhoneNumber",
        when(
            # YOUR CODE HERE: condition -- length is 12 AND starts with "91"
            lit(True),
            concat(lit("+91-"), col("PhoneNumber").substr(3, 10))
        ).otherwise(col("PhoneNumber"))
    )

remediated_df.select("CustomerID", "PhoneNumber").show(5, truncate=False)

## Phase D — The Quarantine Decision: REGISTERED_UNDER_18

Look at the actual flagged rows before deciding anything.

In [ ]:
remediated_df \
    .withColumn("_dob", to_date(col("DateOfBirth"), "yyyy-MM-dd")) \
    .withColumn("_reg", to_date(col("RegistrationDate"), "yyyy-MM-dd")) \
    .withColumn("_age_at_reg", floor(datediff(col("_reg"), col("_dob")) / 365.25).cast("int")) \
    .filter(col("_age_at_reg") < 18) \
    .select("CustomerID", "DateOfBirth", "RegistrationDate", "_age_at_reg") \
    .orderBy("_age_at_reg") \
    .show(20)

**Q: Unlike the email fix, why can't we "correct" these rows instead of quarantining them?**

*Your answer:* _______________

> Hint: with the email issue, we knew the correct value — we were just cleaning its format. With age, we cannot assume the DOB is wrong; the customer may genuinely have registered while under 18 because the form had no age validation at the time. GlobalMart requires 18+ (legal compliance, payment authorization) — these rows must not reach Silver as valid customers.

## Phase E — Final DQ Check + Split Clean vs. Quarantine

In [ ]:
dq_df = remediated_df \
    .withColumn("_dob_temp",   to_date(col("DateOfBirth"),      "yyyy-MM-dd")) \
    .withColumn("_reg_temp",   to_date(col("RegistrationDate"), "yyyy-MM-dd")) \
    .withColumn("_age_at_reg", floor(datediff(col("_reg_temp"), col("_dob_temp")) / 365.25)) \
    .withColumn("_dq_issue",
        when(col("CustomerID").isNull(),                                lit("NULL_CUSTOMER_ID"))
        .when(col("FirstName").isNull() | (trim(col("FirstName")) == ""), lit("NULL_FIRST_NAME"))
        .when(col("Email").isNull(),                                    lit("NULL_EMAIL"))
        .when(~col("Email").rlike(EMAIL_REGEX),                         lit("INVALID_EMAIL_FORMAT"))
        .when(col("_age_at_reg") < 18,                                  lit("REGISTERED_UNDER_18"))
        .otherwise(lit(None))
    )

# TASK 4: split into clean_df (no issue) and quarantine_df (issue present).
# Drop the temp working columns from clean_df; keep _dq_issue on quarantine_df.
clean_df      = dq_df.filter(col("_dq_issue").isNull()) \
                     .drop("_dq_issue", "_dob_temp", "_reg_temp", "_age_at_reg")
quarantine_df = dq_df.filter(col("_dq_issue").isNotNull()) \
                     .drop("_dob_temp", "_reg_temp", "_age_at_reg")

print(f"Total rows  : {bronze_df.count():,}")
print(f"Clean rows  : {clean_df.count():,}")
print(f"Quarantine  : {quarantine_df.count():,}")
quarantine_df.groupBy("_dq_issue").count().orderBy("count", ascending=False).show()

**Q: Do clean rows + quarantine rows add up to the total Bronze row count? Confirm below.**

*Your answer:* _______________

## Phase F — Write Clean + Quarantine Tables

In [ ]:
# Standardize column names + add SCD2 tracking columns before writing to Silver.
# This part is given (not a blank) -- it's infrastructure, not the DQ lesson.
# Gold's dimension build (Day 6) expects exactly these lowercase names to
# exist on gbmart.silver.customers: customer_sk, customer_id, full_name,
# email, phone_number, is_current, effective_start_date, effective_end_date.
# Same surrogate-key pattern used for silver.products in HOL 2.
silver_customers_df = clean_df \
    .withColumnRenamed("CustomerID", "customer_id") \
    .withColumnRenamed("FirstName", "first_name") \
    .withColumnRenamed("LastName", "last_name") \
    .withColumnRenamed("Email", "email") \
    .withColumnRenamed("PhoneNumber", "phone_number") \
    .withColumnRenamed("DateOfBirth", "date_of_birth") \
    .withColumnRenamed("RegistrationDate", "registration_date") \
    .withColumnRenamed("PreferredPaymentMethodID", "preferred_payment_method_id") \
    .withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name"))) \
    .withColumn("effective_start_date", current_date()) \
    .withColumn("effective_end_date", lit(None).cast(DateType())) \
    .withColumn("is_current", lit(True)) \
    .withColumn("customer_sk", sha2(concat_ws("|", col("customer_id"), col("effective_start_date").cast("string")), 256)) \
    .select("customer_sk", "customer_id", "full_name", "first_name", "last_name",
            "email", "phone_number", "date_of_birth", "registration_date",
            "preferred_payment_method_id", "is_current",
            "effective_start_date", "effective_end_date")

print(f"Standardized columns: {silver_customers_df.columns}")

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

# TASK 5: write silver_customers_df to SILVER_TABLE (append mode) and quarantine_df to
# QUARANTINE_TABLE (append mode, with overwriteSchema=true since this table
# may not exist yet). Add a _silver_updated_at / _quarantine_ts timestamp to each.

silver_customers_df.withColumn("_silver_updated_at", current_timestamp()) \
    .write.format("delta").mode("append").saveAsTable(SILVER_TABLE)

quarantine_df.withColumn("_quarantine_ts", current_timestamp()) \
    .write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable(QUARANTINE_TABLE)

print(f"{SILVER_TABLE}     : {spark.table(SILVER_TABLE).count():,} rows")
print(f"{QUARANTINE_TABLE} : {spark.table(QUARANTINE_TABLE).count():,} rows")

## Phase F.5 — Enforce With Delta Constraints

A DQ scan is a check you have to remember to run each time. A Delta constraint is enforced by the table itself, forever, no matter which notebook writes to it next.

The DQ scan above already **guarantees** every row in `clean_df` (and therefore in `silver_customers_df`) has a non-null `customer_id` and a non-null, correctly-formatted `email` — that is precisely why those rows are in `clean_df` and not `quarantine_df`. Turning that guarantee into a table constraint costs one line each, and it means a future bug (in this notebook or a completely different one) can never silently reintroduce a null into either column.

In [ ]:
# TASK 6: add two constraints to gbmart.silver.customers:
#   1. NOT NULL on customer_id
#   2. NOT NULL on email
#   3. a CHECK constraint 'valid_email_shape' requiring email LIKE '%@%.%'
# All three are safe to add here -- clean_df already guarantees them, so this
# is locking in a fact that's already true, not testing an unproven one.

spark.sql(f"ALTER TABLE {SILVER_TABLE} ALTER COLUMN customer_id SET NOT NULL")
spark.sql(f"ALTER TABLE {SILVER_TABLE} ALTER COLUMN email SET NOT NULL")
spark.sql(f"""
    ALTER TABLE {SILVER_TABLE}
    ADD CONSTRAINT valid_email_shape CHECK (email LIKE '%@%.%')
""")

print(f"Constraints added on {SILVER_TABLE}:")
print("  NOT NULL : customer_id")
print("  NOT NULL : email")
print("  CHECK    : valid_email_shape  (email LIKE '%@%.%')")
print()
print("From here on, ANY notebook that tries to append a null customer_id/email,")
print("or a value that doesn't even look like an email, gets its write rejected --")
print("not just this one, and not just today.")

## Phase G — Verify

In [ ]:
df = spark.table(SILVER_TABLE)
print(f"silver.customers rows : {df.count():,}")

# Every remaining email should now pass validation
still_bad = df.filter(~col("email").rlike(EMAIL_REGEX)).count()
print(f"Remaining invalid emails in Silver: {still_bad}  (expected: 0)")

# Phone format check
df.select("customer_id", "phone_number").show(5, truncate=False)

---
## Submission Checklist

```
Submission Checklist
----------------------------------------------------------
[ ] DQ scan completed -- all 5 issue categories tagged correctly
[ ] Email fix applied and re-validated (0 remaining invalid)
[ ] Phone number reformatted to +91-XXXXXXXXXX
[ ] REGISTERED_UNDER_18 quarantine decision justified in your own words
[ ] Clean + quarantine row counts sum to the Bronze total
[ ] silver.customers written and verified
[ ] silver.customers_quarantine written and verified
[ ] NOT NULL (customer_id, email) + CHECK (valid_email_shape) constraints added
── Total Bronze rows:            ______
── Clean rows written:           ______
── Quarantined rows:             ______
----------------------------------------------------------
```